# Civil War News — Notebook 2: Sentiment Analysis

Runs transformer-based sentiment on the cleaned corpus produced by Notebook 1.
LSD and VADER scores are pre-computed in Notebook 1 and embedded in `news_proc`.

**Prerequisites:** Run Notebook 1 first (or have `news_proc/` on Drive).

**Runtime:** GPU required (A100 recommended for 8M+ articles).

**Storage note (USE_DRIVE=False):** `news_proc/` must be in `/content/` from this
same session. The output `sentiment_results.csv` is also ephemeral — download it
at the end via the Download cell.

**Output:** `sentiment_results.csv` in `BASE_PATH`.

## Section 1: Install Packages & Check GPU

In [ ]:
%pip install --q \
    datasets \
    transformers \
    accelerate \
    openpyxl \
    pandas \
    huggingface_hub
print('Packages installed')

In [ ]:
import torch
if torch.cuda.is_available():
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {VRAM_GB:.1f} GB')
    DEVICE = 0
else:
    print('No GPU detected. Go to Runtime > Change runtime type > T4 GPU')
    VRAM_GB = 0
    DEVICE  = -1

## Section 2: Configuration

Edit sentiment flags and storage settings. Nothing below the divider needs changing.


In [ ]:
# ======================================================================
# SENTIMENT METHODS
RUN_TRANSFORMER = True    # Transformer-based (see TRANSFORMER_MODEL below)

# Transformer model options:
#   'distilbert'    distilbert-base-uncased-finetuned-sst-2-english  (fast 2-class baseline)
#   'cardiffnlp'    cardiffnlp/twitter-roberta-base-sentiment-latest  (CSS field standard, 3-class)
#   'siebert'       siebert/sentiment-roberta-large-english            (high accuracy, 2-class)
#   'zero-shot-nli' facebook/bart-large-mnli via zero-shot-classification (most flexible)
TRANSFORMER_MODEL = 'siebert'
ZSL_LABELS = ['positive', 'negative', 'neutral']

# ======================================================================
# STORAGE
USE_DRIVE  = True   # True: Google Drive (persistent); False: /content/ (ephemeral)
DRIVE_BASE = '/content/drive/MyDrive/CivilWarNews'
LOCAL_BASE = '/content/data'
HF_REPO    = 'patrickjcrawford/civil-war-news'

# ======================================================================
# (No edits needed below this line)
import os
BASE_PATH      = DRIVE_BASE if USE_DRIVE else LOCAL_BASE
NEWS_PROC_PATH = f'{LOCAL_BASE}/news_proc'   # always ephemeral — loaded from zip
SENTIMENT_CSV  = f'{BASE_PATH}/sentiment_results.csv'

print(f'Storage    : {"Google Drive" if USE_DRIVE else "Colab /content/data/ (ephemeral)"}')
print(f'Input      : {NEWS_PROC_PATH}')
print(f'Output     : {SENTIMENT_CSV}')
print(f'HF Repo    : hf://datasets/{HF_REPO}')
print(f'Transformer: {TRANSFORMER_MODEL}  (LSD/VADER pre-computed in Notebook 1)')

## Section 3: Storage Setup

Validates required files and uploads `lsd_texts/` if needed.
Required files (only when `RUN_LSD=True`): `lsdSentiment.py` + 4 word-list `.txt` files.


In [ ]:
import os, zipfile
from huggingface_hub import HfApi, login, hf_hub_download

# ── Hugging Face login ─────────────────────────────────────────────────────
try:
    from google.colab import userdata as _ud
    login(token=_ud.get('HF_TOKEN'), add_to_git_credential=False)
    print('[OK] Logged in to Hugging Face.')
except Exception as _hf_e:
    print(f'HF login: {_hf_e}')
    print('Add HF_TOKEN to Colab Secrets (key icon in sidebar).')

def _is_hf_dataset(path):
    return (os.path.exists(f'{path}/dataset_info.json') or
            os.path.exists(f'{path}/dataset_dict.json'))

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    os.makedirs(BASE_PATH, exist_ok=True)
    print(f'Drive mounted.')
else:
    os.makedirs(LOCAL_BASE, exist_ok=True)

os.makedirs(LOCAL_BASE, exist_ok=True)
_drive_zip = f'{DRIVE_BASE}/news_proc.zip'

if not _is_hf_dataset(NEWS_PROC_PATH):
    if USE_DRIVE and os.path.exists(_drive_zip):
        print(f'Extracting news_proc.zip → {LOCAL_BASE} ...')
        with zipfile.ZipFile(_drive_zip, 'r') as _zf:
            _zf.extractall(LOCAL_BASE)
        print('Extracted.')
    else:
        if USE_DRIVE:
            print(f'Not found on Drive: {_drive_zip}')
        print(f'Downloading news_proc.zip from HuggingFace ({HF_REPO}) ...')
        try:
            _hf_zip = hf_hub_download(repo_id=HF_REPO, filename='news_proc.zip',
                                      repo_type='dataset')
            print(f'Extracting from HF cache → {LOCAL_BASE} ...')
            with zipfile.ZipFile(_hf_zip, 'r') as _zf:
                _zf.extractall(LOCAL_BASE)
            print('Extracted.')
        except Exception as _e:
            raise FileNotFoundError(
                f'news_proc.zip not found on Drive or HF ({HF_REPO}). '
                f'Run Notebook 1 first.'
            ) from _e

if not _is_hf_dataset(NEWS_PROC_PATH):
    raise FileNotFoundError(f'Dataset not found: {NEWS_PROC_PATH}')
print('[OK] news_proc found')

## Section 4: Load Data

In [ ]:
from datasets import load_from_disk

news_proc = load_from_disk(NEWS_PROC_PATH)
print(f'Loaded {len(news_proc):,} articles')
print('Columns:', news_proc.column_names)

# Generator — streams from Arrow one row at a time; avoids duplicating
# all article text as a Python list (~3-4 GB for 8M articles).
# Safe because each pipeline branch consumes it exactly once.
def _articles():
    for row in news_proc:
        t = row['article']
        yield t if isinstance(t, str) else ''


## Section 5: Transformer Sentiment

Selected by `TRANSFORMER_MODEL` in the Configuration cell.

| Key | Model | Output |
|-----|-------|--------|
| `distilbert` | distilbert-base-uncased-finetuned-sst-2-english | 2-class |
| `cardiffnlp` | cardiffnlp/twitter-roberta-base-sentiment-latest | 3-class |
| `siebert` | siebert/sentiment-roberta-large-english | 2-class |
| `zero-shot-nli` | facebook/bart-large-mnli | custom labels |

All scores normalized to `transformer_score` in [−1, 1].

In [ ]:
import torch

transformer_scores = {}

if RUN_TRANSFORMER:
    from transformers import pipeline as hf_pipeline

    _TCFGS = {
        'distilbert': {
            'task': 'sentiment-analysis',
            'model': 'distilbert-base-uncased-finetuned-sst-2-english',
            'label_map': {'POSITIVE': 1, 'NEGATIVE': -1},
        },
        'cardiffnlp': {
            'task': 'sentiment-analysis',
            'model': 'cardiffnlp/twitter-roberta-base-sentiment-latest',
            'label_map': None,
        },
        'siebert': {
            'task': 'sentiment-analysis',
            'model': 'siebert/sentiment-roberta-large-english',
            'label_map': {'POSITIVE': 1, 'NEGATIVE': -1},
        },
        'zero-shot-nli': {
            'task': 'zero-shot-classification',
            'model': 'facebook/bart-large-mnli',
            'label_map': None,
        },
    }
    tcfg = _TCFGS[TRANSFORMER_MODEL]

    # Select dtype: BF16 on A100/H100 (native support), FP16 on T4/L4, FP32 on CPU
    if DEVICE >= 0:
        if torch.cuda.is_bf16_supported():
            T_DTYPE, dtype_name = torch.bfloat16, 'bfloat16'
        else:
            T_DTYPE, dtype_name = torch.float16, 'float16'
    else:
        T_DTYPE, dtype_name = torch.float32, 'float32'

    # fp16/bf16 halves memory per tensor — double the float32 batch size baseline
    _half = T_DTYPE != torch.float32
    t_batch = (256 if VRAM_GB >= 30 else (128 if VRAM_GB >= 16 else 32)) if _half \
         else (128 if VRAM_GB >= 30 else (64  if VRAM_GB >= 16 else 16))
    if TRANSFORMER_MODEL == 'siebert':       t_batch = max(t_batch // 2, 8)
    if TRANSFORMER_MODEL == 'zero-shot-nli': t_batch = max(t_batch // 4, 4)

    print(f'Loading: {tcfg["model"]}  dtype={dtype_name}  batch_size={t_batch}')
    pipe = hf_pipeline(tcfg['task'], model=tcfg['model'], device=DEVICE,
                       truncation=True, max_length=512, dtype=T_DTYPE)

    t_labels, t_scores = [], []

    if TRANSFORMER_MODEL == 'zero-shot-nli':
        results = pipe(_articles(), candidate_labels=ZSL_LABELS, batch_size=t_batch)
        for r in results:
            t_labels.append(r['labels'][0])
            sm = dict(zip(r['labels'], r['scores']))
            t_scores.append(float(sm.get(ZSL_LABELS[0], 0) - sm.get(ZSL_LABELS[1], 0)))
    elif TRANSFORMER_MODEL == 'cardiffnlp':
        results = pipe(_articles(), batch_size=t_batch)
        for r in results:
            t_labels.append(r['label'])
            lbl = r['label'].lower()
            sc  = float(r['score'])
            t_scores.append(sc if lbl == 'positive' else (-sc if lbl == 'negative' else 0.0))
    else:
        results = pipe(_articles(), batch_size=t_batch)
        lmap = tcfg['label_map']
        for r in results:
            t_labels.append(r['label'])
            t_scores.append(float(r['score']) * lmap.get(r['label'], 1))

    transformer_scores = {
        'transformer_label': t_labels,
        'transformer_score': t_scores,
        'transformer_model': [TRANSFORMER_MODEL] * len(t_labels),
    }
    mean_sc = sum(t_scores) / len(t_scores)
    print(f'Transformer done. Mean score: {mean_sc:.4f}')
    cnts = {}
    for l in t_labels: cnts[l] = cnts.get(l, 0) + 1
    print('Label distribution:', cnts)
else:
    transformer_scores = {
        'transformer_label': [''] * len(news_proc),
        'transformer_score': [float('nan')] * len(news_proc),
        'transformer_model': [''] * len(news_proc),
    }
    print('Transformer skipped (RUN_TRANSFORMER=False)')


## Section 6: Save Results

In [ ]:
import pandas as pd

lsd_cols   = ['lsd_pos', 'lsd_neg', 'lsd_pos_share', 'lsd_neg_share', 'lsd_neu_share',
               'lsd_relpropdiff', 'lsd_abspropdiff', 'lsd_logit', 'lsd_asinh']
vader_cols = ['vader_sent', 'vader_pos', 'vader_neg', 'vader_neu']

# Pull pre-computed LSD/VADER columns from news_proc (computed in Notebook 1)
avail      = [c for c in lsd_cols + vader_cols if c in news_proc.column_names]
missing_sv = [c for c in lsd_cols + vader_cols if c not in news_proc.column_names]
if missing_sv:
    print(f'WARNING: sentiment columns missing from news_proc (re-run Notebook 1?): {missing_sv}')

sent_df = pd.DataFrame({
    'article_id': news_proc['article_id'],
    'lccn':       news_proc['lccn'],
    'year':       news_proc['year'],
    **{col: news_proc[col] for col in avail},
})
for col, vals in transformer_scores.items():
    sent_df[col] = vals

sent_df.to_csv(SENTIMENT_CSV, index=False)
print(f'Saved {len(sent_df):,} rows x {len(sent_df.columns)} columns')
print(f'File: {SENTIMENT_CSV}')
print(sent_df.describe(include='all').round(4).to_string())

In [ ]:
from huggingface_hub import HfApi
import os

if os.path.exists(SENTIMENT_CSV):
    _api = HfApi()
    _api.create_repo(HF_REPO, private=True, repo_type='dataset', exist_ok=True)
    try:
        _existing = list(_api.list_repo_files(HF_REPO, repo_type='dataset'))
    except Exception:
        _existing = []
    _fname = os.path.basename(SENTIMENT_CSV)
    if _fname in _existing:
        print(f'[already on HF] {_fname}')
    else:
        _sz = os.path.getsize(SENTIMENT_CSV) / 1e6
        print(f'Uploading {_fname} to {HF_REPO} ({_sz:.1f} MB) ...')
        _api.upload_file(
            path_or_fileobj=SENTIMENT_CSV,
            path_in_repo=_fname,
            repo_id=HF_REPO,
            repo_type='dataset',
        )
        print(f'Saved: hf://datasets/{HF_REPO}/{_fname}')
else:
    print(f'File not found: {SENTIMENT_CSV}')